# Phase 5 - CLIP Embeddings, ChromaDB and Metadata Export

This notebook runs Phase 5 on Kaggle. It generates image embeddings from representative crops, writes embedding metadata, and prepares outputs for future phases.

In [ ]:
!pip install -q torch torchvision numpy pillow tqdm open_clip_torch
!pip install -q chromadb

In [ ]:
import os
import sys
from pathlib import Path

ROOT = '/kaggle/working'
os.chdir(ROOT)

project_dir = Path('/kaggle/working/AI_Video_Intelligence')
if not project_dir.exists():
    !git clone https://github.com/your-user/AI_Video_Intelligence.git /kaggle/working/AI_Video_Intelligence
    os.chdir(project_dir)
else:
    os.chdir(project_dir)

sys.path.insert(0, str(project_dir))
print('Working directory:', project_dir)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import json
import numpy as np

# Resolve inputs
phase4_dir = Path('outputs/phase4')
phase5_dir = Path('outputs/phase5')
phase5_dir.mkdir(parents=True, exist_ok=True)

track_metadata_file = phase4_dir / 'track_metadata.json'
if not track_metadata_file.exists():
    raise FileNotFoundError(f'Missing phase 4 metadata: {track_metadata_file}')

with open(track_metadata_file, 'r', encoding='utf-8') as f:
    track_metadata = json.load(f)

print('Loaded track metadata count:', len(track_metadata))
print('First record keys:', sorted(track_metadata[0].keys()) if track_metadata else [])

In [ ]:
from ai.embeddings.embedding_pipeline import EmbeddingPipeline

crops_dir = Path('outputs/phase3/production_runs/test/04_representative_selection/crops')
if not crops_dir.exists():
    raise FileNotFoundError(f'Missing representative crops folder: {crops_dir}')

pipeline = EmbeddingPipeline()
pipeline.generate_embeddings(
    crops_directory=crops_dir,
    output_directory=phase5_dir,
)

In [ ]:
from pathlib import Path
import json

embedding_meta = phase5_dir / 'embedding_metadata.json'
embeddings_np = phase5_dir / 'image_embeddings.npy'

print('embedding_metadata exists:', embedding_meta.exists())
print('image_embeddings exists:', embeddings_np.exists())

if embedding_meta.exists():
    with open(embedding_meta, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print('metadata_count:', len(data))
    if data:
        sample = data[0]
        print('sample keys:', sorted(sample.keys()))
        for key in ['start_time_seconds', 'end_time_seconds', 'duration_seconds', 'start_timestamp', 'end_timestamp', 'timestamp']:
            print(key, sample.get(key))

## Optional: create a simple ChromaDB-compatible index folder

This helps later phases if they expect a local database directory.

In [ ]:
from pathlib import Path
chroma_dir = Path('outputs/phase6/chromadb')
chroma_dir.mkdir(parents=True, exist_ok=True)
print('Chroma directory ready:', chroma_dir)